# Adaptive steps

A PROTOTYPE of the compute gate (Prompt 268, COMPUTE_GATE_II.md): it is not in the catalogue and not shipped. It reads its input as the text `gravitas_input` and its settings as the JSON text `gravitas_params`, and returns `gravitas_output`, a dict of numbers. Python only, numpy and scipy only.


In [ ]:
import json, math
import numpy as np
from scipy.integrate import solve_ivp
doc = json.loads(gravitas_input)
assert doc.get("format") == "gravitas.orbital-system" and doc.get("version") == 1, "this notebook reads gravitas.orbital-system/1"
init = doc.get("initial")
assert isinstance(init, dict), "the file has no initial state: save it again from the Orbital System Builder"
G = float(init["G"])
bodies = init["bodies"]
m = np.array([b["mass"] for b in bodies], float)
X0 = np.array([[b["x"], b["y"]] for b in bodies], float)
V0 = np.array([[b["vx"], b["vy"]] for b in bodies], float)
N = len(m)
assert 2 <= N <= 12, "from two to twelve bodies"
d0 = np.linalg.norm(X0[:, None, :] - X0[None, :, :], axis=2)
iu = np.triu_indices(N, 1)
assert d0[iu].min() > 1e-9 * d0[iu].max(), "refused: two bodies start at the same place"
k = int(np.argmin(d0[iu])); ia, ja = int(iu[0][k]), int(iu[1][k])
Ts = 2 * math.pi * math.sqrt(d0[ia, ja] ** 3 / (G * (m[ia] + m[ja])))
com = (m[:, None] * X0).sum(axis=0) / m.sum()
r_far = float(np.max(np.linalg.norm(X0 - com, axis=1)))

def accel(x):
    d = x[None, :, :] - x[:, None, :]
    r2 = (d ** 2).sum(axis=2)
    np.fill_diagonal(r2, 1.0)
    w = r2 ** -1.5
    np.fill_diagonal(w, 0.0)
    return G * ((m[None, :] * w)[:, :, None] * d).sum(axis=1)

def rhs(t, y):
    return np.concatenate([y[2 * N:], accel(y[:2 * N].reshape(N, 2)).ravel()])

ref = solve_ivp(rhs, (0, Ts), np.concatenate([X0.ravel(), V0.ravel()]), method="DOP853", rtol=1e-13, atol=1e-13)
Xref = ref.y[:2 * N, -1].reshape(N, 2)
def final_error(x):
    return float(np.max(np.linalg.norm(x - Xref, axis=1)))

In [ ]:
A = [[], [1 / 5], [3 / 40, 9 / 40], [44 / 45, -56 / 15, 32 / 9],
     [19372 / 6561, -25360 / 2187, 64448 / 6561, -212 / 729],
     [9017 / 3168, -355 / 33, 46732 / 5247, 49 / 176, -5103 / 18656],
     [35 / 384, 0, 500 / 1113, 125 / 192, -2187 / 6784, 11 / 84]]
C = [0, 1 / 5, 3 / 10, 4 / 5, 8 / 9, 1, 1]
BSTAR = [5179 / 57600, 0, 7571 / 16695, 393 / 640, -92097 / 339200, 187 / 2100, 1 / 40]
EC = [a - b for a, b in zip(A[6] + [0], BSTAR)]
MAX_STEPS = 200000

def f(t, y):
    return rhs(t, y)

def integrate(tol):
    y = np.concatenate([X0.ravel(), V0.ravel()])
    t, h = 0.0, 1e-3 * Ts
    acc = rej = nfev = 0
    f0 = f(0.0, y); nfev += 1
    while t < Ts:
        if acc + rej >= MAX_STEPS:
            raise ValueError("refused: more than %d steps; the system has a close encounter at this tolerance" % MAX_STEPS)
        if h < 1e-12 * Ts:
            raise ValueError("refused: the step fell below 1e-12 of the span")
        h = min(h, Ts - t)
        ks = [f0]
        for i in range(1, 7):
            ks.append(f(0.0, y + h * sum(a * kj for a, kj in zip(A[i - 1], ks))))
        nfev += 6
        ynew = y + h * sum(a * kj for a, kj in zip(A[6], ks))
        err = h * sum(e * kj for e, kj in zip(EC, ks))
        scale = tol + tol * np.abs(y)
        en = float(np.sqrt(np.mean((err / scale) ** 2)))
        if en <= 1.0:
            t += h; y = ynew; acc += 1
            f0 = ks[6]
        else:
            rej += 1
        h *= min(5.0, max(0.2, 0.9 * (en ** -0.2 if en > 0 else 5.0)))
    return y, acc, rej, nfev

def rk4_error(n):
    h = Ts / n
    x, v = X0.copy(), V0.copy()
    for _ in range(n):
        k1x, k1v = v, accel(x)
        k2x, k2v = v + 0.5 * h * k1v, accel(x + 0.5 * h * k1x)
        k3x, k3v = v + 0.5 * h * k2v, accel(x + 0.5 * h * k2x)
        k4x, k4v = v + h * k3v, accel(x + h * k3x)
        x, v = (x + h / 6 * (k1x + 2 * k2x + 2 * k3x + k4x), v + h / 6 * (k1v + 2 * k2v + 2 * k3v + k4v))
    return final_error(x)

out = {}
for tol in [1e-4, 1e-6, 1e-8, 1e-10]:
    y, acc, rej, nfev = integrate(tol)
    e = final_error(y[:2 * N].reshape(N, 2))
    n = 50
    while n <= 102400 and rk4_error(n) > e:
        n *= 2
    out[repr(tol)] = {"err_rel": e / r_far, "accepted": acc, "rejected": rej, "nfev": nfev, "rk4_nfev": (4 * n if n <= 102400 else None)}
gravitas_output = {"answer": {"Ts": Ts, "r_far": r_far, "tolerances": out}}